# Adidas Supply Planning Agent — Colab Runner

This notebook clones the [buying-planning-agent](https://github.com/isalgi/buying-planning-agent) repo and runs it on Google Colab, using Google Gemini's free API tier (no OpenAI billing required).

**Note on state:** Colab runtimes are ephemeral — the SQLite database and FAISS index reset every time this runtime restarts or disconnects. This is fine for testing/demos, not for persistent use.

Run the cells below in order.

In [ ]:
# 1. Clone the repository
%cd /content
!rm -rf buying-planning-agent
!git clone https://github.com/isalgi/buying-planning-agent.git
%cd buying-planning-agent

In [ ]:
# 2. Install dependencies
# pydantic<2.11 is required: newer pydantic breaks gradio 4.44.1's API schema introspection
!pip install -q -r requirements.txt
!pip install -q "pydantic<2.11"

## Get a free Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Click "Create API key" and copy it.
3. Run the next cell **on its own** (not via "Run all") and wait for the input box to appear at the top before pasting — it validates the key format immediately so you don't hit a confusing error two cells later.

In [ ]:
# 3. Configure environment variables
import getpass

while True:
    google_api_key = getpass.getpass("Enter your Google AI Studio API key: ").strip()
    if not google_api_key:
        print("\u274c Empty input \u2014 the key box may not have been ready. Try again.")
        continue
    if not google_api_key.startswith("AIza"):
        print(f"\u26a0\ufe0f  That doesn't look like a Gemini key (expected it to start with 'AIza'). "
              f"Got something starting with '{google_api_key[:6]}'. Try again, or press Ctrl+C if you're sure it's correct.")
        continue
    break

env_content = f"""GOOGLE_API_KEY={google_api_key}
LANGSMITH_API_KEY=your_key_here
LANGCHAIN_PROJECT=adidas-supply-planning
LANGCHAIN_TRACING_V2=false
LANGSMITH_TRACING=False
LANGSMITH_ENDPOINT=https://api.smith.langchain.com/
LANGSMITH_PROJECT=adidas-supply-planning
"""

with open(".env", "w") as f:
    f.write(env_content)

print(f"\u2713 .env created (key length: {len(google_api_key)}, starts with: {google_api_key[:6]})")

(Optional) If you also want LangSmith tracing, get a free key at https://smith.langchain.com/ (Settings → API Keys) and re-run the cell above, replacing `your_key_here` and setting the tracing flags to `true`/`True`.

In [ ]:
# 3b. Sanity-check the key actually works before initializing anything
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
try:
    _resp = _client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": "Say OK"}],
    )
    print("\u2713 Key works:", _resp.choices[0].message.content)
except Exception as e:
    print("\u274c Key test failed:", e)
    print("Re-run the previous cell and paste the key again.")

In [ ]:
# 4. Initialize the database and RAG (FAISS) index
!python db.py
!python rag.py

In [ ]:
# 5. Launch the Gradio app
# share=True is required on Colab since 127.0.0.1 isn't reachable from your browser —
# Gradio will print a public *.gradio.live link instead.
from gradio_ui import demo

demo.launch(share=True)

## Notes

- **Rate limits:** Gemini's free tier caps `gemini-2.5-flash` at 5 requests/minute. Space out test queries by a few seconds.
- **Restarting:** If you restart the Colab runtime, re-run all cells from the top — the cloned repo, `.env`, database, and FAISS index are all wiped.
- **Stopping the app:** Use Runtime → Interrupt execution to stop the Gradio server.